# 01 · Extracción del ASG del IMSS
Descarga los cortes mensuales de asegurados (datos abiertos del IMSS), filtra y guarda en parquet.
Reutiliza las funciones de `extraer_asg.py` del repo, sin modificarlo.

In [5]:
# También hay que instalar el requirements.txt
import sys
from pathlib import Path
from datetime import date
import polars as pl


RAIZ = Path.cwd().parent # Ruta raíz del proyecto es una carpeta arriba
sys.path.insert(0, str(RAIZ))

import extraer_asg as ea # Esta es la función que exporta los datos del imss del repo original




In [6]:
print("Dimensiones:", ea.DIMENSIONES)
print("Métricas:", ea.METRICAS)
print("Parámetro de filtro -> columna:")
for parametro, col in ea.PARAMETROS_FILTRO.items():
    print(f"  {parametro:20s} -> {col}")

Dimensiones: ['cve_delegacion', 'cve_subdelegacion', 'cve_entidad', 'cve_municipio', 'sector_economico_1', 'sector_economico_2', 'sector_economico_4', 'tamano_patron', 'sexo', 'rango_edad', 'rango_salarial', 'rango_uma', 'ptpd']
Métricas: ['asegurados', 'no_trabajadores', 'ta', 'teu', 'tec', 'tpu', 'tpc', 'ta_sal', 'teu_sal', 'tec_sal', 'tpu_sal', 'tpc_sal', 'masa_sal_ta', 'masa_sal_teu', 'masa_sal_tec', 'masa_sal_tpu', 'masa_sal_tpc']
Parámetro de filtro -> columna:
  SECTOR_1             -> sector_economico_1
  SECTOR_2             -> sector_economico_2
  SECTOR_4             -> sector_economico_4
  ENTIDADES            -> cve_entidad
  MUNICIPIOS           -> cve_municipio
  DELEGACIONES         -> cve_delegacion
  SUBDELEGACIONES      -> cve_subdelegacion
  SEXO                 -> sexo
  TAMANO_PATRON        -> tamano_patron
  RANGO_EDAD           -> rango_edad
  RANGO_SALARIAL       -> rango_salarial
  RANGO_UMA            -> rango_uma
  PLATAFORMA_DIGITAL   -> ptpd


In [9]:
# Actividad: 
act = pl.read_csv(RAIZ / "catalogos" / "actividades_imss.csv")
print(act.group_by("nivel").len().sort("nivel"))
act.filter(pl.col("nivel") == "sector_economico_2")

shape: (3, 2)
┌────────────────────┬─────┐
│ nivel              ┆ len │
│ ---                ┆ --- │
│ str                ┆ u32 │
╞════════════════════╪═════╡
│ sector_economico_1 ┆ 10  │
│ sector_economico_2 ┆ 62  │
│ sector_economico_4 ┆ 276 │
└────────────────────┴─────┘


nivel,clave,descripcion
str,i64,str
"""sector_economico_2""",1,"""Agricultura"""
"""sector_economico_2""",2,"""Ganadería"""
"""sector_economico_2""",3,"""Silvicultura"""
"""sector_economico_2""",4,"""Pesca"""
"""sector_economico_2""",5,"""Caza"""
…,…,…
"""sector_economico_2""",91,"""Servicios de Enseñanza, Invest…"
"""sector_economico_2""",92,"""Servicios Médicos, Asistencia …"
"""sector_economico_2""",93,"""Agrupaciones Mercantiles, Prof…"


## Variables del ASG y parámetros de extracción

### Cada fila
Casa fila es una combinación de características (estado, municipio, sexo, edad, rango salarial, ...) con el total de puestos y salarios que tiene esa combinación.


### Dimensiones (13)
| Variable | Descripción (diccionario IMSS) | Valores |
|---|---|---|
| `cve_delegacion` | Delegación de adscripción al IMSS | Numérico |
| `cve_subdelegacion` | Subdelegación de adscripción al IMSS | Numérico |
| `cve_entidad` | Entidad federativa (catálogo IMSS) | 1–32, mismo orden que INEGI |
| `cve_municipio` | Municipio (catálogo IMSS, **no** INEGI) | Alfanumérico, p. ej. `A01` |
| `sector_economico_1` | División | 0–9 (2 y 3 = transformación) |
| `sector_economico_2` | Grupo | 62 claves, p. ej. `38` |
| `sector_economico_4` | Fracción | 276 claves, p. ej. `3803` |
| `tamano_patron` | Tamaño del registro patronal | S1 = 1 puesto · S2 = 2–5 · S3 = 6–50 · S4 = 51–250 · S5 = 251–500 · S6 = 501–1,000 · S7 = más de 1,000 · NA |
| `sexo` | Sexo | 1 hombre · 2 mujer · 3 no binario |
| `rango_edad` | Rango de edad | E1 = menos de 15 · E2 = 15–19 · … (de 5 en 5) · E14 = 75 o más · NA |
| `rango_salarial` | Salario en veces el salario mínimo | W1 = hasta 1 · W2 = más de 1 hasta 2 · … · W25 = más de 24 hasta 25 · NA |
| `rango_uma` | Salario en veces la UMA (desde febrero de 2017) | W1–W25, igual que arriba pero en UMA · NA |
| `ptpd` | Trabajador de plataforma digital | 0 no · 1 sí |

### Métricas (17)
| Variable | Descripción |
|---|---|
| `asegurados` | Número de asegurados |
| `no_trabajadores` | Asegurados sin un empleo asociado |
| `ta` | **Puestos de trabajo afiliados**: la cifra de empleo formal del IMSS |
| `tpu`, `teu` | Puestos permanentes / eventuales urbanos |
| `tpc`, `tec` | Puestos permanentes / eventuales del campo |
| `ta_sal` (y `tpu_sal`, `teu_sal`, `tpc_sal`, `tec_sal`) | Puestos con un salario asociado |
| `masa_sal_ta` (y `masa_sal_tpu`, `masa_sal_teu`, `masa_sal_tpc`, `masa_sal_tec`) | Masa salarial asociada a esos puestos (suma de salarios base de cotización diarios) |

Notas:
- `ta = tpu + teu + tpc + tec`. Cuenta puestos, no personas. Si alguien tiene dos jefes cuenta dos veces. 
- Salario promedio = `masa_sal_ta / ta` (esto hace el repo original, que es todo el salario dividido entre # de puestos afiliados). Puede ser también `masa_sal_ta / ta_sal` y solo dividir por puestos con salario asociado. Decisión que tomar (yo me guío por esta).
- Rangos en salarios mínimos: no son comparables entre años, porque el mínimo subió mucho más que la inflación. 

### Parámetros de `extraer_asg.py`

**Salida y periodo**
| Parámetro | Qué hace | Ejemplo |
|---|---|---|
| `NOMBRE` | Nombra carpetas y archivos de salida | `"autopartes"` |
| `CARPETA_SALIDA` | Dónde se guarda (relativa al repo o absoluta) | `"data"` |
| `FECHA_INI`, `FECHA_FIN` | Periodo; `FECHA_FIN = None` es el último mes cerrado | `date(2018, 1, 1)` |

**Filtros**: cada uno es un vector de claves o `False` (sin filtro). Entre filtros se combinan con **Y**; dentro de un vector, con **O**.
| Parámetro | Columna | Ejemplo |
|---|---|---|
| `SECTOR_1` | `sector_economico_1` | `[3]` |
| `SECTOR_2` | `sector_economico_2` | `[38]` (toma todas las 38xx) |
| `SECTOR_4` | `sector_economico_4` | `[3802, 3803]` |
| `ENTIDADES` | `cve_entidad` | `[11, 14]` |
| `MUNICIPIOS` | `cve_municipio` | `["A01"]`; combínalo con `ENTIDADES` (el código `Y44` existe en Puebla y en Sonora) |
| `DELEGACIONES`, `SUBDELEGACIONES` | `cve_delegacion`, `cve_subdelegacion` | `[39]` |
| `SEXO` | `sexo` | `[2]` |
| `TAMANO_PATRON` | `tamano_patron` | `["S5", "S6", "S7"]` |
| `RANGO_EDAD` | `rango_edad` | `["E3", "E4"]` |
| `RANGO_SALARIAL` | `rango_salarial` | `["W1"]` |
| `RANGO_UMA` | `rango_uma` | `["W1", "W2", "W3"]` |
| `PLATAFORMA_DIGITAL` | `ptpd` | `[1]` |

**Agregación**
| Parámetro | Qué hace |
|---|---|
| `NIVEL_SECTOR` | Nivel al que se suman los paneles: `"auto"` (el más fino con filtro), `"SECTOR_1"`, `"SECTOR_2"` o `"SECTOR_4"` |
| `ETIQUETAS` | Crea la columna `grupo` juntando claves, p. ej. `{"nivel": "SECTOR_4", "grupos": {"autopartes": [3802, 3803]}}`; `None` = no crearla |
| `AGREGAR_POR` | Columnas del panel agregado; `"SECTOR"` = el nivel de `NIVEL_SECTOR` |
| `METRICAS_PANEL` | Métricas que se suman en el panel |
| `PANEL_CLAVE`, `PANEL_POR`, `PANEL_METRICA` | Configuración de `panel_ocupados.py` |

**Descarga**
| Parámetro | Qué hace | Default |
|---|---|---|
| `N_HILOS` | Meses que se descargan a la vez | 4 |
| `TIMEOUT_SIN_DATOS` | Segundos sin recibir datos antes de cortar | 60 |
| `MAX_SEG_INTENTO` | Tope de segundos por intento | 180 |
| `REINTENTOS` | Intentos por mes (un 404 no se reintenta) | 5 |

Vamos a dejar todos los parámetros en un diccionario para poder ajustarlos solo desde esta celda siguiente. Los valores default de la función filtran fracciones de autorpartes.

In [11]:
PARAMS = {
    # Salida y periodo
    "NOMBRE": "completo",
    "CARPETA_SALIDA": "data",   # se extrae dentro del repo 
    "FECHA_INI": date(2018, 1, 1),
    "FECHA_FIN": None,                                   # None = último mes cerrado

    # Filtros de actividad: vector de claves o False
    "SECTOR_1": False,
    "SECTOR_2": False,
    "SECTOR_4": False,
    "NIVEL_SECTOR": "SECTOR_2", # con esto el repo suma por clave a 4 dígitos. 

    # Otros filtros: vector o False
    "ENTIDADES": False,
    "MUNICIPIOS": False,
    "DELEGACIONES": False,
    "SUBDELEGACIONES": False,
    "SEXO": False,
    "TAMANO_PATRON": False,
    "RANGO_EDAD": False,
    "RANGO_SALARIAL": False,
    "RANGO_UMA": False,
    "PLATAFORMA_DIGITAL": False,

    # Agregación
    "ETIQUETAS": None,
    "AGREGAR_POR": ["fecha", "SECTOR", "ENTIDADES", "MUNICIPIOS"],
    "METRICAS_PANEL": ["ta", "tpu", "teu", "ta_sal", "masa_sal_ta"],   # ta_sal para el salario correcto
    "PANEL_CLAVE": "SECTOR",
    "PANEL_POR": ["ENTIDADES", "MUNICIPIOS"],
    "PANEL_METRICA": "ta",

    # Descarga
    "N_HILOS": 2,
    "TIMEOUT_SIN_DATOS": 60,
    "MAX_SEG_INTENTO": 300,
    "REINTENTOS": 5,
}

Los parámetros del notebook reemplazan a los de `extraer_asg.py`, y `cargar_config()` los valida y arma los filtros.

In [ ]:
# Usamos nuestros parámetros

ea.DEFAULTS.update(PARAMS)    # nuestros parámetros reemplazan a los del .py
cfg = ea.cargar_config()      # el repo los valida y arma cfg["FILTROS"]
cfg["FILTROS"]

{}

## Carpetas de salida
Vamos a definir carpetas para cada mes que extraigamos y así tener organizados los datos


In [16]:
carpeta_salida = ea.ruta_salida(cfg)
carpeta_mes = carpeta_salida / f"asg_{cfg['NOMBRE']}_meses"
fechas = ea.fechas_corte(cfg["FECHA_INI"], cfg["FECHA_FIN"])

print(f"{len(fechas)} meses: {fechas[0]} a {fechas[-1]}")

106 meses: 2018-01-31 a 2026-10-31


## Descarga
Cada mes se baja, se filtra y se guarda como `asg_AAAA-MM-DD.parquet` en `carpeta_mes`. Los meses que ya existen se saltan, así que si se corta, basta con volver a correr la celda.

In [ ]:
import tempfile


# Un mes de prueba

carpeta_mes.mkdir(parents=True, exist_ok=True)
ea.revisar_firma(carpeta_mes, cfg["FILTROS"])

with tempfile.TemporaryDirectory() as tmp:
    print(ea.procesar(fechas[-1], tmp, cfg, carpeta_mes))

('2026-10-31', 'NO DISPONIBLE (HTTP Error 403: Forbidden)')


In [18]:
# todas las fechas 

with tempfile.TemporaryDirectory() as tmp:
    for f in fechas:
        print(ea.procesar(f, tmp, cfg, carpeta_mes))

('2018-01-31', 'NO DISPONIBLE (HTTP Error 403: Forbidden)')
('2018-02-28', 'NO DISPONIBLE (HTTP Error 403: Forbidden)')
('2018-03-31', 'NO DISPONIBLE (HTTP Error 403: Forbidden)')
('2018-04-30', 'NO DISPONIBLE (HTTP Error 403: Forbidden)')
('2018-05-31', 'NO DISPONIBLE (HTTP Error 403: Forbidden)')
('2018-06-30', 'NO DISPONIBLE (HTTP Error 403: Forbidden)')
('2018-07-31', 'NO DISPONIBLE (HTTP Error 403: Forbidden)')
('2018-08-31', 'NO DISPONIBLE (HTTP Error 403: Forbidden)')
('2018-09-30', 'NO DISPONIBLE (HTTP Error 403: Forbidden)')
('2018-10-31', 'NO DISPONIBLE (HTTP Error 403: Forbidden)')
('2018-11-30', 'NO DISPONIBLE (HTTP Error 403: Forbidden)')
('2018-12-31', 'NO DISPONIBLE (HTTP Error 403: Forbidden)')
('2019-01-31', 'NO DISPONIBLE (HTTP Error 403: Forbidden)')
('2019-02-28', 'NO DISPONIBLE (HTTP Error 403: Forbidden)')
('2019-03-31', 'NO DISPONIBLE (HTTP Error 403: Forbidden)')
('2019-04-30', 'NO DISPONIBLE (HTTP Error 403: Forbidden)')
('2019-05-31', 'NO DISPONIBLE (HTTP Erro

KeyboardInterrupt: 